In [1]:
from google.colab import drive
import pandas as pd

In [ ]:
drive.mount('/content/gdrive')

Mounted at /content/gdrive


In [ ]:
df = pd.read_csv('/content/gdrive/MyDrive/Faculdade IA/Terceiro Semestre/Aprendizado de máquina/abt_churn.csv')

In [ ]:
for i in df.columns:
  print(i)

dtRef
idUsuario
qtdeTransacoes
qtdeDias
mediaTransacoesDias
saldoPontos
qtdePontosPos
qtdePontosNeg
qtdeDiasUltimaTransacao
qtdeDiasPrimeiraTransacao
qtdSkuDistintos
qtdeChatMessage
qtdePresença
qtdeTrocaStreamElements
qtdeChurn
qtdePonei
qtdeAirflowLover
qtdePresencaStreak
qtdeDailyLoot
qtdeRLover
qtdeVendaItemRPG
qtdeTransacoesD7
qtdeDiasD7
saldoPontosD7
qtdePontosPosD7
qtdePontosNegD7
qtdeTransacoesD14
qtdeDiasD14
saldoPontosD14
qtdePontosPosD14
qtdePontosNegD14
qtdeTransacoesD28
qtdeDiasD28
saldoPontosD28
qtdePontosPosD28
qtdePontosNegD28
propAvgQtdeTransacoes
propAvgQtdeDias
propAvgMediaTransacoesDias
propAvgSaldoPontos
propAvgQtdePontosPos
propAvgQtdePontosNeg
flagChurn


**SAMPLE**

Nesta etapa vamos ordenar a base de dados conforme a data de referencia.

In [ ]:
df = df.sort_values(['dtRef'])

Agora iremos definir uma data de corte para guardarmos os dados do ano atual para testarmos posteriormente (A data será a partir de 01/01/2025)

In [ ]:
df_oot = df[df['dtRef'] >= '2025-03-01'].copy()
df_model = df[df['dtRef'] < '2025-03-01'].copy()

Separação de Feature e Target

In [ ]:
X = df_model.drop(columns=['flagChurn'])
y = df_model['flagChurn']

Aplicação do Split para termos Train/Test

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

**EXPLORE**

Nesta etapa faremos a análise bivariado para identificar as médias de cada variável para cada classe de Target

In [ ]:
df_analise = X_train.copy()
df_analise['target'] = y_train
resumo_churn = df_analise.groupby(['target']).mean(numeric_only=True).T

In [ ]:
resumo_churn.sort_values(by=[0])

In [ ]:
resumo_churn.sort_values(by=[1])

Aplicaremos os dataframes de Treino em um modelo de árvore de decisão para identificar as variáveis que impactam no Churn

In [ ]:
from sklearn.tree import DecisionTreeClassifier

In [ ]:
arvore = DecisionTreeClassifier(max_depth=4)
arvore.fit(X_train.drop(columns=['dtRef','idUsuario']),y_train)

DecisionTreeClassifier(max_depth=4)

Verificando as variáveis importantes encontradas na árvore

In [ ]:
importances = pd.Series(arvore.feature_importances_, index=X_train.drop(columns=['dtRef','idUsuario']).columns)

In [ ]:
print(importances.sort_values(ascending=False).head(5))

qtdeDiasD14                0.695843
qtdeDiasUltimaTransacao    0.104802
propAvgQtdeDias            0.101055
qtdeDiasD28                0.038877
qtdeTransacoesD7           0.021857
dtype: float64


Verificamos se há valores nulos na base de dados. Encontramos que não há dados faltantes.

In [ ]:
X_train.isnull().sum()

**MODIFY**

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

Este comando seleciona as variáveis númericas, excluindo Data e ID de usuário

In [ ]:
num_cols = X_train.select_dtypes(exclude='object').columns

Esse código fará uma padronização dos dados através do ColumnTransformer. Apliquei dois tipos: Com scaler para usarmos no modelo de Regressão e sem aplicar nenhum pré-processamento, apenas irá eliminar as colunas que não são as númericas

In [ ]:
preprocessor_scaler = ColumnTransformer([
    ('num', StandardScaler(), num_cols)
],remainder='drop')

preprocessor_no_scaler = ColumnTransformer([
    ('num', 'passthrough', num_cols)
],remainder='drop')

**MODEL**

Preparei o Pipeline do modelo de Random Forest e Regressão Logística

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV

In [ ]:
pipeline_forest = Pipeline([('preprocessing',preprocessor_no_scaler),
                          ('model',RandomForestClassifier(random_state=42))])

In [ ]:
pipeline_reg = Pipeline([('preprocessing',preprocessor_scaler),
                         ('model',LogisticRegression())])

Seleção dos parâmetros que iremos testar para ambos os modelos

In [ ]:
parametros_forest = {
    'model__n_estimators': [100,200], #Variável de número de árvores que o modelo irá gerar, permitindo uma melhor generalização
    'model__max_depth': [5,10] #Controla a profundidade de cada árvore
}

parametros_reg = {
    'model__C': [0.001, 0.01, 0.1, 1, 10], #Variável de regularização -> Técnica usada para evitar que o modelo entre em Overfitting
    'model__penalty': ['l1','l2'], #Tipo da regularização -> l1 = Pode zerar coeficientes | l2 = Mais estável
    'model__solver': ['liblinear'], #O algoritmo que encontra os coeficientes, neste caso será usado o liblinear por lidar bem com classificação binária
    'model__class_weight': [None, 'balanced'] #É usado o balanced se os dados estiverem desbalanceados (mais de uma classe do que outra)
}

Estruturando o Grid de cada modelo, aplicando o pipeline + parametros para posteriormente treinarmos os modelos com os dados de treino

In [ ]:
grid_forest = GridSearchCV(
    pipeline_forest, parametros_forest, cv=3, scoring='roc_auc'
)

grid_reg = GridSearchCV(
    pipeline_reg, parametros_reg, cv=3, scoring='roc_auc'
)

In [ ]:
grid_forest.fit(X_train, y_train)

GridSearchCV(cv=3,
             estimator=Pipeline(steps=[('preprocessing',
                                        ColumnTransformer(transformers=[('num',
                                                                         'passthrough',
                                                                         Index(['qtdeTransacoes', 'qtdeDias', 'mediaTransacoesDias', 'saldoPontos',
       'qtdePontosPos', 'qtdePontosNeg', 'qtdeDiasUltimaTransacao',
       'qtdeDiasPrimeiraTransacao', 'qtdSkuDistintos', 'qtdeChatMessage',
       'qtdePresença', 'qtdeTrocaStreamElements', 'q...
       'qtdeTransacoesD28', 'qtdeDiasD28', 'saldoPontosD28',
       'qtdePontosPosD28', 'qtdePontosNegD28', 'propAvgQtdeTransacoes',
       'propAvgQtdeDias', 'propAvgMediaTransacoesDias', 'propAvgSaldoPontos',
       'propAvgQtdePontosPos', 'propAvgQtdePontosNeg'],
      dtype='object'))])),
                                       ('model', RandomForestClassifier())]),
             param_grid={'model__max_depth': [5, 10],
                         'model__n_estimators': [100, 200]},
             scoring='roc_auc')

In [ ]:
grid_reg.fit(X_train, y_train)

GridSearchCV(cv=3,
             estimator=Pipeline(steps=[('preprocessing',
                                        ColumnTransformer(transformers=[('num',
                                                                         StandardScaler(),
                                                                         Index(['qtdeTransacoes', 'qtdeDias', 'mediaTransacoesDias', 'saldoPontos',
       'qtdePontosPos', 'qtdePontosNeg', 'qtdeDiasUltimaTransacao',
       'qtdeDiasPrimeiraTransacao', 'qtdSkuDistintos', 'qtdeChatMessage',
       'qtdePresença', 'qtdeTrocaStreamElements'...
       'qtdePontosPosD28', 'qtdePontosNegD28', 'propAvgQtdeTransacoes',
       'propAvgQtdeDias', 'propAvgMediaTransacoesDias', 'propAvgSaldoPontos',
       'propAvgQtdePontosPos', 'propAvgQtdePontosNeg'],
      dtype='object'))])),
                                       ('model', LogisticRegression())]),
             param_grid={'model__C': [0.001, 0.01, 0.1, 1, 10],
                         'model__class_weight': [None, 'balanced'],
                         'model__penalty': ['l1', 'l2'],
                         'model__solver': ['liblinear']},
             scoring='roc_auc')

**ASSESS**

In [ ]:
import pickle
from sklearn.metrics import roc_auc_score, accuracy_score

Predições do Y teste e OOT para ambos os modelos

In [ ]:
y_oot = df_oot['flagChurn']
X_oot = df_oot.drop(columns=['flagChurn'])

In [ ]:
y_prob_train_forest = grid_forest.predict_proba(X_train)[:,1]
y_prob_teste_forest = grid_forest.predict_proba(X_test)[:,1]
y_prob_oot_forest = grid_forest.predict_proba(X_oot)[:,1]

In [ ]:
y_prob_train_reg = grid_reg.predict_proba(X_train)[:,1]
y_prob_teste_reg = grid_reg.predict_proba(X_test)[:,1]
y_prob_oot_reg = grid_reg.predict_proba(X_oot)[:,1]

Visualização das métricas ROC AUC por Treino, Teste e OOT

In [ ]:
print(f'Forest | ROC AUC | TREINO: {roc_auc_score(y_train, y_prob_train_forest):.4f}')
print(f'Regressão | ROC AUC | TREINO: {roc_auc_score(y_train, y_prob_train_reg):.4f}')
print(f'Forest | ACURÁCIA | TREINO: {accuracy_score(y_train, grid_forest.predict(X_train)):.4f}')
print(f'Regressão | ACURÁCIA | TREINO: {accuracy_score(y_train, grid_reg.predict(X_train)):.4f}')
print('')
print(f'Forest | ROC AUC | TESTE: {roc_auc_score(y_test, y_prob_teste_forest):.4f}')
print(f'Regressão | ROC AUC | TESTE: {roc_auc_score(y_test, y_prob_teste_reg):.4f}')
print(f'Forest | ACURÁCIA | TESTE: {accuracy_score(y_test, grid_forest.predict(X_test)):.4f}')
print(f'Regressão | ACURÁCIA | TESTE: {accuracy_score(y_test, grid_reg.predict(X_test)):.4f}')
print('')
print(f'Forest | ROC AUC | OOT: {roc_auc_score(y_oot, y_prob_oot_forest):.4f}')
print(f'Regressão | ROC AUC | OOT: {roc_auc_score(y_oot, y_prob_oot_reg):.4f}')
print(f'Forest | ACURÁCIA | OOT: {accuracy_score(y_oot, grid_forest.predict(X_oot)):.4f}')
print(f'Regressão | ACURÁCIA | OOT: {accuracy_score(y_oot, grid_reg.predict(X_oot)):.4f}')

Forest | ROC AUC | TREINO: 0.8438
Regressão | ROC AUC | TREINO: 0.8179
Forest | ACURÁCIA | TREINO: 0.7614
Regressão | ACURÁCIA | TREINO: 0.7411

Forest | ROC AUC | TESTE: 0.8167
Regressão | ROC AUC | TESTE: 0.8167
Forest | ACURÁCIA | TESTE: 0.7366
Regressão | ACURÁCIA | TESTE: 0.7281

Forest | ROC AUC | OOT: 0.8435
Regressão | ROC AUC | OOT: 0.8390
Forest | ACURÁCIA | OOT: 0.7745
Regressão | ACURÁCIA | OOT: 0.7759


**RESULTADOS**: Vemos que os modelos conseguiram rankear corretamente churn vs não churn, na maioira dos casos, considerando a métrica de ROC AUC (Com leve vantagem para o modelo RandomForest).
As 3 bases funcionaram bem e ficaram com valores próximos, indicando que não houve um Overfitting.

Partiremos para a análise de Lifts/Gains, preparando a base de Testes para analisarmos quantos dos potenciais churns nós detectaremos com os modelos

In [ ]:
base_teste = X_test.copy()
base_teste['target'] = y_test

total_churn = base_teste['target'].sum()

base_teste_forest = base_teste.copy()
base_teste_reg = base_teste.copy()

base_teste_forest['proba_forest'] = y_prob_teste_forest
base_teste_reg['proba_reg'] = y_prob_teste_reg

base_teste_forest = base_teste_forest.sort_values(by=['proba_forest'], ascending=False)
base_teste_reg = base_teste_reg.sort_values(by=['proba_reg'], ascending=False)

Nesta etapa, iremos analisar os 10%,20%,30% das maiores porcentagens dos modelos e verificar como eles se saíram.

In [ ]:
resultados = []
bases = (base_teste_forest, base_teste_reg)
for base in bases:
  lista_ref = []
  for p in [0.1, 0.2, 0.3]:

      cutoff = int(len(base) * p)

      top = base.iloc[:cutoff]

      churn_capturados = top['target'].sum()

      perc_capturado = churn_capturados / total_churn

      lift = perc_capturado / p

      lista_ref.append({
          'Percentual Base': f'{int(p*100)}%',
          'Churn Capturado': round(perc_capturado*100,2),
          'Lift': round(lift,2)
      })
  resultados.append(lista_ref)

In [ ]:
resultado_forest = pd.DataFrame(resultados[0])
resultado_reg = pd.DataFrame(resultados[1])

Para o modelo Forest, ele conseguiu capturar 18.16% dos Churns totais da base selecionando os 10% que estão no topo, resultando em um Lift de 1.82x. Por outro lado, o modelo de regressão linear atingiu 17.93% com Lift de 1.79x. Ambos modelos tiveram um bom desempenho, mas o Forest ganha com uma leve vantagem.

In [ ]:
print(resultado_forest)

  Percentual Base  Churn Capturado  Lift
0             10%            18.16  1.82
1             20%            34.94  1.75
2             30%            49.20  1.64


In [ ]:
print(resultado_reg)

  Percentual Base  Churn Capturado  Lift
0             10%            17.93  1.79
1             20%            34.48  1.72
2             30%            50.34  1.68


FALTA COLOCAR AS DESCRIÇÕES ACIMA E EXPORTAR O ARQUIVO PICKLE, DEPOIS SUBIR PRO GITHUB

https://drive.google.com/file/d/1raLBNs-rr_G83J_HYlxR0G_Nf87SreiW/view

Por fim, fazemos a serialização para exportar o arquivo .pkl.
Neste caso, escolhi o que apresentou o melhor ROC e o melhor Lift que foi o modelo do Random Forest.

In [ ]:
with open('detector_de_churn_forest_v1.pkl', 'wb') as file:
  pickle.dump(grid_forest.best_estimator_, file)

In [ ]:
#Para importar o arquivo .pkl seguir o código abaixo
with open('detector_de_churn_forest_v1.pkl', 'rb') as f:
    modelo = pickle.load(f)